# Reading Raw Data

In [0]:
df = spark.table("bronze_layer.vehicles_table")

# Data Transformation

In [0]:
df.display()

In [0]:
from pyspark.sql.functions import col, to_timestamp

df =df \
    .withColumn("crash_unit_id", col("crash_unit_id").cast("string")) \
    .withColumn("crash_record_id", col("crash_record_id").cast("string")) \
    .withColumn("crash_date", to_timestamp("crash_date")) \
    .withColumn("unit_no", col("unit_no").cast("int")) \
    .withColumn("unit_type", col("unit_type").cast("string")) \
    .withColumn("vehicle_id", col("vehicle_id").cast("int")) \
    .withColumn("make", col("make").cast("string")) \
    .withColumn("model", col("model").cast("string")) \
    .withColumn("lic_plate_state", col("lic_plate_state").cast("string")) \
    .withColumn("vehicle_year", col("vehicle_year").cast("int")) \
    .withColumn("vehicle_defect", col("vehicle_defect").cast("string")) \
    .withColumn("vehicle_type", col("vehicle_type").cast("string")) \
    .withColumn("vehicle_use", col("vehicle_use").cast("string")) \
    .withColumn("travel_direction", col("travel_direction").cast("string")) \
    .withColumn("maneuver", col("maneuver").cast("string")) \
    .withColumn("occupant_cnt", col("occupant_cnt").cast("int")) \
    .withColumn("first_contact_point", col("first_contact_point").cast("string")) \
    .withColumn("num_passengers", col("num_passengers").cast("int")) \
    .withColumn("axle_cnt", col("axle_cnt").cast("int")) \
    .withColumn("gvwr", col("gvwr").cast("string")) \
    .withColumn("usdot_no", col("usdot_no").cast("string")) \
    .withColumn("ccmc_no", col("ccmc_no").cast("string")) \
    .withColumn("ilcc_no", col("ilcc_no").cast("string")) \
    .withColumn("un_no", col("un_no").cast("string")) \
    .withColumn("trailer1_width", col("trailer1_width").cast("string")) \
    .withColumn("trailer1_length", col("trailer1_length").cast("string")) \
    .withColumn("total_vehicle_length", col("total_vehicle_length").cast("string")) \
    .withColumn("trailer2_width", col("trailer2_width").cast("string")) \
    .withColumn("trailer2_length", col("trailer2_length").cast("string"))

## Fix Date datatype

In [0]:
from pyspark.sql.types import * 
from pyspark.sql.functions import * 




## Trimming

In [0]:
#trim the string columns
# normalization 

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name,trim(col(field.name)))


    

    


In [0]:
from pyspark.sql.functions import sum 
cols_to_keep = []
df_nulls = df.select([
    sum(col(c).isNull().cast('int')).alias(c)
    for c in df.columns
]).collect()[0].asDict()






for key, value in df_nulls.items():
    if value < 60000:
        cols_to_keep.append(key)





df = df.select(cols_to_keep)




# Remove & Cleaning Nulls

In [0]:
string_nulls = ["", " ", "NA", "N/A", "null", "None"]
numeric_nulls = [0, -1, 9999]
df_clean =df

df.dtypes

for column, data_type in df_clean.dtypes:
    if data_type == 'string':
        df_clean = df_clean.fillna({column: 'UNKNOWN'}) 
        
    elif data_type in ("int", "bigint", "double", "float"):
        df_clean = df_clean.fillna({column: 0}) 

         
        








# WRITE TO CLEAN SILVER TABLE

In [0]:
df_clean.write \
    .mode('overwrite') \
    .format("delta") \
    .save("abfss://source@datalakeimewore.dfs.core.windows.net/ChicagoCrashes/silver/vehicles")